In [1]:
import feedparser
import pandas as pd
import time
import requests
import os
print("tout est good")

tout est good


In [2]:

# Configuration
nom_fichier = "bulletins_anssi_avec_cves.csv"
url_avis = "https://www.cert.ssi.gouv.fr/avis/feed/"
url_alertes = "https://www.cert.ssi.gouv.fr/alerte/feed/"

# --- BLOC 2 : FONCTIONS ---
def extraire_flux_rss(url, type_bulletin):
    flux = feedparser.parse(url)
    liste_bulletins = []
    for entry in flux.entries:
        liste_bulletins.append({
            "id_anssi": entry.link.strip("/").split("/")[-1],
            "titre": entry.title,
            "description": entry.description,
            "date_publication": entry.published,
            "lien": entry.link,
            "type_bulletin": type_bulletin
        })
    time.sleep(1)
    return liste_bulletins


In [3]:
def extraire_cves_du_bulletin(lien_bulletin):
    url_json = f"{lien_bulletin.strip('/')}/json/"
    try:
        response = requests.get(url_json)
        if response.status_code == 200:
            data = response.json()
            return [item['name'] for item in data.get("cves", [])]
        return []
    except Exception as e:
        print(f"Erreur pour {lien_bulletin}: {e}")
        return []

In [4]:
def mettre_a_jour_donnees(nom_fichier, url_avis, url_alertes):
    print("Récupération des flux...")
    avis = extraire_flux_rss(url_avis, "Avis")
    alertes = extraire_flux_rss(url_alertes, "Alerte")
    df_flux_actuel = pd.DataFrame(avis + alertes)
    df_flux_actuel['id_anssi'] = df_flux_actuel['id_anssi'].astype(str).str.strip()

    if os.path.exists(nom_fichier):
        df_existant = pd.read_csv(nom_fichier)
        df_existant['id_anssi'] = df_existant['id_anssi'].astype(str).str.strip()
        
        # On cherche ce qui est dans le flux mais PAS dans le CSV
        mask = ~df_flux_actuel['id_anssi'].isin(df_existant['id_anssi'])
        nouveaux_items = df_flux_actuel[mask].copy()
        
        if not nouveaux_items.empty:
            print(f"{len(nouveaux_items)} manquants détectés. Extraction des CVE...")
            nouveaux_items['liste_cves'] = nouveaux_items['lien'].apply(extraire_cves_du_bulletin)
            df_final = pd.concat([df_existant, nouveaux_items], ignore_index=True)
            df_final = df_final.drop_duplicates(subset=['id_anssi'])
            df_final.to_csv(nom_fichier, index=False, encoding="utf-8-sig")
            print(f"Mise à jour réussie. Total : {len(df_final)}")
        else:
            print("Aucun nouveau bulletin à ajouter.")
            df_final = df_existant
    else:
        print("Fichier inexistant, création complète du CSV...")
        df_flux_actuel['liste_cves'] = df_flux_actuel['lien'].apply(extraire_cves_du_bulletin)
        df_final = df_flux_actuel.drop_duplicates(subset=['id_anssi'])
        df_final.to_csv(nom_fichier, index=False, encoding="utf-8-sig")

    print(f"Nombre total de bulletins dans le CSV : {len(df_final)}")
    display(df_final.tail())
    return df_final

In [5]:
df_final = mettre_a_jour_donnees(nom_fichier, url_avis, url_alertes)

Récupération des flux...
Fichier inexistant, création complète du CSV...
Nombre total de bulletins dans le CSV : 80


,id_anssi,titre,description,date_publication,lien,type_bulletin,liste_cves
75,CERTFR-2026-ALE-001,[MàJ] Multiples vulnérabilités dans Ivanti End...,[Mise à jour du 09 février 2026] Le 6 février ...,"Fri, 30 Jan 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"[CVE-2026-1340, CVE-2026-1281]"
76,CERTFR-2026-ALE-002,[MàJ] Vulnérabilité dans Cisco Catalyst SD-WAN...,Une vulnérabilité a été découverte dans Cisco ...,"Wed, 25 Feb 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"[CVE-2026-20127, CVE-2022-20775]"
77,CERTFR-2026-ALE-003,Note d’alerte – Ciblage des messageries instan...,Les travaux conjoints des services membres du ...,"Fri, 20 Mar 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[]
78,CERTFR-2026-ALE-004,Vulnérabilité dans F5 BIG-IP Access Policy Man...,"Le 15 octobre 2025, F5 a publié un avis de séc...","Tue, 31 Mar 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[CVE-2025-53521]
79,CERTFR-2026-ALE-005,[Màj] Vulnérabilité dans Microsoft Exchange Se...,"[Mise à jour du 11 juin 2026] Le 9 juin 2026, ...","Fri, 15 May 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[CVE-2026-42897]


In [6]:

print("Aperçu :")
display(df_final.head())


Aperçu :


,id_anssi,titre,description,date_publication,lien,type_bulletin,liste_cves
0,CERTFR-2026-AVI-0699,Vulnérabilité dans Cisco Catalyst SD-WAN (05 j...,Une vulnérabilité a été découverte dans Cisco ...,"Fri, 05 Jun 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-...,Avis,[CVE-2026-20245]
1,CERTFR-2026-AVI-0711,Multiples vulnérabilités dans les VPN Check Po...,De multiples vulnérabilités ont été découverte...,"Tue, 09 Jun 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-...,Avis,"[CVE-2026-50752, CVE-2026-50751]"
2,CERTFR-2026-AVI-0712,Vulnérabilité dans Veeam Backup & Replication ...,Une vulnérabilité a été découverte dans Veeam ...,"Tue, 09 Jun 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-...,Avis,[CVE-2026-44963]
3,CERTFR-2026-AVI-0713,Vulnérabilité dans les produits Schneider Elec...,Une vulnérabilité a été découverte dans les pr...,"Tue, 09 Jun 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-...,Avis,[CVE-2026-8045]
4,CERTFR-2026-AVI-0714,Multiples vulnérabilités dans les produits Sie...,De multiples vulnérabilités ont été découverte...,"Tue, 09 Jun 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-...,Avis,"[CVE-2025-15467, CVE-2026-24349, CVE-2025-40808]"


In [7]:
# Test pour vérifier le update

# 1. On supprime les 5 dernières lignes pour simuler un fichier incomplet
if os.path.exists(nom_fichier):
    df_test = pd.read_csv(nom_fichier)
    taille_avant = len(df_test)
    df_test = df_test.iloc[:-5] 
    df_test.to_csv(nom_fichier, index=False, encoding="utf-8-sig")
    print(f"Test : {taille_avant} -> {len(df_test)} lignes. (Les 5 dernières supprimées)")
    display(df_test.tail(7))
    
    # 2. On lance la mise à jour pour réparer le fichier
    print("\n--- Lancement de la mise à jour (Réparation) ---")
    df_final = mettre_a_jour_donnees(nom_fichier, url_avis, url_alertes)
    
    # 3. Vérification finale
    print("\n--- Vérification finale des données ---")
    print(f"Taille finale après update : {len(df_final)} lignes.")
    print("Voici les dernières lignes (les 5 que nous avions supprimées doivent être revenues) :")
    display(df_final.tail(7))
else:
    print("Fichier introuvable, impossible de réaliser le test.")

Test : 80 -> 75 lignes. (Les 5 dernières supprimées)


,id_anssi,titre,description,date_publication,lien,type_bulletin,liste_cves
68,CERTFR-2025-ALE-008,[MàJ] Vulnérabilité dans Roundcube (05 juin 2025),[Mise à jour du 06 juin 2025] Une preuve de co...,"Thu, 05 Jun 2025 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,['CVE-2025-49113']
69,CERTFR-2025-ALE-009,Multiples vulnérabilités dans Citrix NetScaler...,**[Mise à jour du 17 juillet 2025]** L'éditeur...,"Tue, 01 Jul 2025 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"['CVE-2025-6543', 'CVE-2025-5777']"
70,CERTFR-2025-ALE-010,[MàJ] Multiples vulnérabilités dans Microsoft ...,**[Mise à jour du 23 juillet 2025]** Le 20 jui...,"Mon, 21 Jul 2025 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"['CVE-2025-53770', 'CVE-2025-53771']"
71,CERTFR-2025-ALE-011,Incidents de sécurité dans les pare-feux Sonic...,"[Mise à jour du 7 août 2025] Le 6 août 2025, S...","Tue, 05 Aug 2025 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[]
72,CERTFR-2025-ALE-012,Vulnérabilité dans Citrix NetScaler ADC et Net...,"Le 26 août 2025, Citrix a publié un bulletin d...","Tue, 26 Aug 2025 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,['CVE-2025-7775']
73,CERTFR-2025-ALE-013,[MàJ] Multiples vulnérabilités dans Cisco ASA ...,**[Mise à jour du 07 novembre 2025]** Le 5 nov...,"Thu, 25 Sep 2025 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"['CVE-2025-20333', 'CVE-2025-20362']"
74,CERTFR-2025-ALE-014,[MàJ] Vulnérabilité dans React Server Componen...,**[Mise à jour du 11 décembre 2025]** Le CERT-...,"Fri, 05 Dec 2025 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"['CVE-2025-55182', 'CVE-2025-66478']"



--- Lancement de la mise à jour (Réparation) ---
Récupération des flux...
5 manquants détectés. Extraction des CVE...
Mise à jour réussie. Total : 80
Nombre total de bulletins dans le CSV : 80


,id_anssi,titre,description,date_publication,lien,type_bulletin,liste_cves
75,CERTFR-2026-ALE-001,[MàJ] Multiples vulnérabilités dans Ivanti End...,[Mise à jour du 09 février 2026] Le 6 février ...,"Fri, 30 Jan 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"[CVE-2026-1340, CVE-2026-1281]"
76,CERTFR-2026-ALE-002,[MàJ] Vulnérabilité dans Cisco Catalyst SD-WAN...,Une vulnérabilité a été découverte dans Cisco ...,"Wed, 25 Feb 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"[CVE-2026-20127, CVE-2022-20775]"
77,CERTFR-2026-ALE-003,Note d’alerte – Ciblage des messageries instan...,Les travaux conjoints des services membres du ...,"Fri, 20 Mar 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[]
78,CERTFR-2026-ALE-004,Vulnérabilité dans F5 BIG-IP Access Policy Man...,"Le 15 octobre 2025, F5 a publié un avis de séc...","Tue, 31 Mar 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[CVE-2025-53521]
79,CERTFR-2026-ALE-005,[Màj] Vulnérabilité dans Microsoft Exchange Se...,"[Mise à jour du 11 juin 2026] Le 9 juin 2026, ...","Fri, 15 May 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[CVE-2026-42897]



--- Vérification finale des données ---
Taille finale après update : 80 lignes.
Voici les dernières lignes (les 5 que nous avions supprimées doivent être revenues) :


,id_anssi,titre,description,date_publication,lien,type_bulletin,liste_cves
73,CERTFR-2025-ALE-013,[MàJ] Multiples vulnérabilités dans Cisco ASA ...,**[Mise à jour du 07 novembre 2025]** Le 5 nov...,"Thu, 25 Sep 2025 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"['CVE-2025-20333', 'CVE-2025-20362']"
74,CERTFR-2025-ALE-014,[MàJ] Vulnérabilité dans React Server Componen...,**[Mise à jour du 11 décembre 2025]** Le CERT-...,"Fri, 05 Dec 2025 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"['CVE-2025-55182', 'CVE-2025-66478']"
75,CERTFR-2026-ALE-001,[MàJ] Multiples vulnérabilités dans Ivanti End...,[Mise à jour du 09 février 2026] Le 6 février ...,"Fri, 30 Jan 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"[CVE-2026-1340, CVE-2026-1281]"
76,CERTFR-2026-ALE-002,[MàJ] Vulnérabilité dans Cisco Catalyst SD-WAN...,Une vulnérabilité a été découverte dans Cisco ...,"Wed, 25 Feb 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,"[CVE-2026-20127, CVE-2022-20775]"
77,CERTFR-2026-ALE-003,Note d’alerte – Ciblage des messageries instan...,Les travaux conjoints des services membres du ...,"Fri, 20 Mar 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[]
78,CERTFR-2026-ALE-004,Vulnérabilité dans F5 BIG-IP Access Policy Man...,"Le 15 octobre 2025, F5 a publié un avis de séc...","Tue, 31 Mar 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[CVE-2025-53521]
79,CERTFR-2026-ALE-005,[Màj] Vulnérabilité dans Microsoft Exchange Se...,"[Mise à jour du 11 juin 2026] Le 9 juin 2026, ...","Fri, 15 May 2026 00:00:00 +0000",https://www.cert.ssi.gouv.fr/alerte/CERTFR-202...,Alerte,[CVE-2026-42897]
